# SpaceX Falcon 9 - Data Wrangling
This notebook cleans the Falcon 9 launch dataset, handles missing payload values, and converts landing outcomes into the binary target used for classification.


In [ ]:
import pandas as pd
import numpy as np

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv'
df = pd.read_csv(url)
df.head()


## Inspect data quality
Before modeling, inspect shape, types, missingness and the distribution of landing outcomes.


In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum().sort_values(ascending=False).head(10))
print(df['Outcome'].value_counts(dropna=False))


## Handle missing values
Payload mass is numeric, so missing values are replaced by the column mean. LandingPad is allowed to remain missing because not every mission has a landing-pad attempt.


In [ ]:
df['PayloadMass'] = df['PayloadMass'].fillna(df['PayloadMass'].mean())


## Create the landing-success class
Unsuccessful/controlled outcomes are assigned class 0; successful landing outcomes are assigned class 1. This converts the business question into a binary classification target.


In [ ]:
bad_outcomes = {
    'False Ocean','False ASDS','None None','False RTLS','None ASDS'
}
df['Class'] = (~df['Outcome'].isin(bad_outcomes)).astype(int)
print(df['Class'].value_counts())
print('Overall landing-success rate:', round(df['Class'].mean()*100, 1), '%')


## Feature engineering
Categorical columns are one-hot encoded so that orbit, launch site, landing pad and booster serial can be used by Scikit-learn models.


In [ ]:
features = df[['FlightNumber','PayloadMass','Orbit','LaunchSite','Flights','GridFins','Reused','Legs','LandingPad','Block','ReusedCount','Serial']].copy()
features = pd.get_dummies(features, columns=['Orbit','LaunchSite','LandingPad','Serial'])
features = features.astype(float)
print('Encoded feature columns:', features.shape[1])
features.to_csv('dataset_part_3.csv', index=False)
df.to_csv('dataset_part_2.csv', index=False)
